In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import TimeSeriesSplit

In [2]:
df = pd.read_csv(
    "../data/spy_modeling_data.csv",
    index_col="Date",
    parse_dates= True
)

HORIZON = 3

dev_df = df.loc[:"2022-12-31"].iloc[:-HORIZON].copy()

final_test_df = df.loc[
    "2023-01-01":
].copy()

print("Development:", dev_df.shape)
print(dev_df.index.min(), "→", dev_df.index.max())

print("\nFinal test:", final_test_df.shape)
print(final_test_df.index.min(), "→", final_test_df.index.max())

Development: (7513, 16)
1993-03-01 00:00:00 → 2022-12-27 00:00:00

Final test: (912, 16)
2023-01-03 00:00:00 → 2026-08-21 00:00:00


In [3]:
tscv = TimeSeriesSplit(
    n_splits=5,
    test_size=504,
    gap=HORIZON
)

for fold, (train_idx, val_idx) in enumerate(
    tscv.split(dev_df),
    start=1
):
    fold_train = dev_df.iloc[train_idx]
    fold_val = dev_df.iloc[val_idx]

    print(f"\nFOLD {fold}")

    print(
        "Train:",
        fold_train.index.min(),
        "→",
        fold_train.index.max()
    )

    print(
        "Validation:",
        fold_val.index.min(),
        "→",
        fold_val.index.max()
    )


FOLD 1
Train: 1993-03-01 00:00:00 → 2012-12-18 00:00:00
Validation: 2012-12-24 00:00:00 → 2014-12-23 00:00:00

FOLD 2
Train: 1993-03-01 00:00:00 → 2014-12-18 00:00:00
Validation: 2014-12-24 00:00:00 → 2016-12-22 00:00:00

FOLD 3
Train: 1993-03-01 00:00:00 → 2016-12-19 00:00:00
Validation: 2016-12-23 00:00:00 → 2018-12-26 00:00:00

FOLD 4
Train: 1993-03-01 00:00:00 → 2018-12-20 00:00:00
Validation: 2018-12-27 00:00:00 → 2020-12-24 00:00:00

FOLD 5
Train: 1993-03-01 00:00:00 → 2020-12-21 00:00:00
Validation: 2020-12-28 00:00:00 → 2022-12-27 00:00:00


In [4]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, roc_auc_score

feature_cols = [
    col for col in dev_df.columns
    if col != "target_3d"
]

In [5]:
tree_fold_results = []

for fold, (train_idx, val_idx) in enumerate(
    tscv.split(dev_df),
    start=1
):
    # Historical train / future validation
    fold_train = dev_df.iloc[train_idx]
    fold_val = dev_df.iloc[val_idx]

    # Separate X and y
    X_fold_train = fold_train[feature_cols]
    y_fold_train = fold_train["target_3d"].astype(int)

    X_fold_val = fold_val[feature_cols]
    y_fold_val = fold_val["target_3d"].astype(int)

    # Fresh model for this historical experiment
    tree = DecisionTreeClassifier(
        max_depth=4,
        min_samples_leaf=50,
        random_state=42
    )

    tree.fit(X_fold_train, y_fold_train)

    # Predictions
    val_pred = tree.predict(X_fold_val)
    val_prob = tree.predict_proba(X_fold_val)[:, 1]

    # Metrics
    accuracy = accuracy_score(
        y_fold_val,
        val_pred
    )

    auc = roc_auc_score(
        y_fold_val,
        val_prob
    )

    tree_fold_results.append({
        "fold": fold,
        "accuracy": accuracy,
        "roc_auc": auc
    })

    print(
        f"Fold {fold} | "
        f"Accuracy: {accuracy:.4f} | "
        f"AUC: {auc:.4f}"
    )

Fold 1 | Accuracy: 0.5794 | AUC: 0.6087
Fold 2 | Accuracy: 0.5952 | AUC: 0.5595
Fold 3 | Accuracy: 0.6210 | AUC: 0.5506
Fold 4 | Accuracy: 0.5079 | AUC: 0.5578
Fold 5 | Accuracy: 0.5536 | AUC: 0.5371


In [6]:
tree_walk_forward = pd.DataFrame(
    tree_fold_results
)

tree_walk_forward

,fold,accuracy,roc_auc
0,1,0.579365,0.608732
1,2,0.595238,0.559495
2,3,0.621032,0.550600
3,4,0.507937,0.557831
4,5,0.553571,0.537097


In [7]:
print(
    "Mean Accuracy:",
    tree_walk_forward["accuracy"].mean()
)
print(
    "Mean ROC-AUC:",
    tree_walk_forward["roc_auc"].mean()
)

Mean Accuracy: 0.5714285714285715
Mean ROC-AUC: 0.5627511613717546


In [8]:
from sklearn.base import clone

def walk_forward_evaluate(model, model_name):
    results = []

    for fold, (train_idx, val_idx) in enumerate(
        tscv.split(dev_df),
        start=1
    ):
        fold_train = dev_df.iloc[train_idx]
        fold_val = dev_df.iloc[val_idx]

        X_train_fold = fold_train[feature_cols]
        y_train_fold = fold_train["target_3d"].astype(int)

        X_val_fold = fold_val[feature_cols]
        y_val_fold = fold_val["target_3d"].astype(int)

        # Fresh model for every fold
        fold_model = clone(model)

        fold_model.fit(
            X_train_fold,
            y_train_fold
        )

        pred = fold_model.predict(X_val_fold)
        prob = fold_model.predict_proba(X_val_fold)[:, 1]

        results.append({
            "model": model_name,
            "fold": fold,
            "accuracy": accuracy_score(y_val_fold, pred),
            "roc_auc": roc_auc_score(y_val_fold, prob)
        })

    return pd.DataFrame(results)

In [9]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (
    RandomForestClassifier,
    HistGradientBoostingClassifier
)
from xgboost import XGBClassifier


log_reg_wf = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=1000))
])

tree_wf = DecisionTreeClassifier(
    max_depth=4,
    min_samples_leaf=50,
    random_state=42
)

rf_wf = RandomForestClassifier(
    n_estimators=300,
    max_depth=6,
    min_samples_leaf=50,
    max_features="sqrt",
    random_state=42,
    n_jobs=-1
)

hgb_wf = HistGradientBoostingClassifier(
    learning_rate=0.05,
    max_iter=200,
    max_leaf_nodes=15,
    min_samples_leaf=50,
    random_state=42
)

xgb_wf = XGBClassifier(
    n_estimators=300,
    learning_rate=0.03,
    max_depth=3,
    min_child_weight=10,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="binary:logistic",
    eval_metric="logloss",
    random_state=42,
    n_jobs=-1
)

In [10]:
all_results = pd.concat([
    walk_forward_evaluate(log_reg_wf, "Logistic Regression"),
    walk_forward_evaluate(tree_wf, "Decision Tree"),
    walk_forward_evaluate(rf_wf, "Random Forest"),
    walk_forward_evaluate(hgb_wf, "HistGradientBoosting"),
    walk_forward_evaluate(xgb_wf, "XGBoost")
], ignore_index=True)

all_results

,model,fold,accuracy,roc_auc
0,Logistic Regression,1,0.573413,0.576823
1,Logistic Regression,2,0.626984,0.569940
2,Logistic Regression,3,0.630952,0.582061
3,Logistic Regression,4,0.521825,0.557682
4,Logistic Regression,5,0.563492,0.547260
5,Decision Tree,1,0.579365,0.608732
6,Decision Tree,2,0.595238,0.559495
7,Decision Tree,3,0.621032,0.550600
8,Decision Tree,4,0.507937,0.557831
9,Decision Tree,5,0.553571,0.537097


In [11]:
summary = (
    all_results
    .groupby("model")
    .agg(
        mean_accuracy=("accuracy", "mean"),
        mean_auc=("roc_auc", "mean"),
        auc_std=("roc_auc", "std")
    )
    .sort_values("mean_auc", ascending=False)
)

summary

,mean_accuracy,mean_auc,auc_std
model,,,
Random Forest,0.578968,0.576580,0.018580
Logistic Regression,0.583333,0.566753,0.014214
Decision Tree,0.571429,0.562751,0.027179
XGBoost,0.570238,0.555751,0.017928
HistGradientBoosting,0.561111,0.546047,0.009752


In [12]:
rf_oof_results = []

for fold, (train_idx, val_idx) in enumerate(
    tscv.split(dev_df),
    start=1
):
    fold_train = dev_df.iloc[train_idx]
    fold_val = dev_df.iloc[val_idx]

    X_fold_train = fold_train[feature_cols]
    y_fold_train = fold_train["target_3d"].astype(int)

    X_fold_val = fold_val[feature_cols]
    y_fold_val = fold_val["target_3d"].astype(int)

    fold_model = clone(rf_wf)

    fold_model.fit(
        X_fold_train,
        y_fold_train
    )

    probabilities = fold_model.predict_proba(
        X_fold_val
    )[:, 1]

    fold_results = pd.DataFrame({
        "actual": y_fold_val,
        "probability": probabilities,
        "fold": fold
    }, index=fold_val.index)

    rf_oof_results.append(fold_results)

In [13]:
rf_oof = pd.concat(rf_oof_results).sort_index()

print(rf_oof.shape)

rf_oof.head()

(2520, 3)


,actual,probability,fold
Date,,,
2012-12-24,0,0.422353,1
2012-12-26,0,0.491315,1
2012-12-27,1,0.480773,1
2012-12-28,1,0.435804,1
2012-12-31,1,0.419747,1


In [14]:
# Checkpoint — Walk-Forward Modeling

## Completed

- Built 15 SPY features:
  - Momentum
  - Trend
  - Volatility
  - Market behavior

- Target:
  - `target_3d = 1` if SPY gains > 0.5% over the next 3 trading days
  - otherwise `0`

- Final modeling dataset:
  - 8,428 observations
  - 15 features
  - 1993-03-01 → 2026-08-21

## Baseline Models Tested

Models:
- DummyClassifier
- Logistic Regression
- Decision Tree
- Random Forest
- HistGradientBoosting
- XGBoost

Single validation results showed only weak predictive signal.

## Walk-Forward Validation

Used expanding-window validation with:

- 5 folds
- ~2 years / 504 trading days per validation fold
- 3-day gap to protect the target horizon
- 2023–2026 kept separate as final test data

Mean walk-forward ROC-AUC:

- Random Forest: ~0.5766
- Logistic Regression: ~0.5668
- Decision Tree: ~0.5628
- XGBoost: ~0.5558
- HistGradientBoosting: ~0.5460

Current primary candidate:
Random Forest

Simple benchmark:
Logistic Regression

## Current Position

Generated Random Forest out-of-fold predictions:

`rf_oof.shape = (2520, 3)`

Columns:

- `actual`
  - true target (0 or 1)

- `probability`
  - Random Forest estimated probability that `target_3d = 1`

- `fold`
  - walk-forward fold that produced the prediction

These are approximately pseudo-live predictions from 2012–2022 because each prediction was generated by a model trained only on earlier data.

## NEXT STEP

Analyze whether higher Random Forest probabilities actually correspond to higher realized success rates.

Example:

0.40–0.45 probability
0.45–0.50
0.50–0.55
0.55–0.60
0.60+

For each probability bucket, inspect:

- number of observations
- average predicted probability
- actual target success rate

Goal:

Determine whether model probability contains useful ranking / trading signal before building the signal engine and backtest.

DO NOT TOUCH 2023–2026 FINAL TEST YET.

SyntaxError: leading zeros in decimal integer literals are not permitted; use an 0o prefix for octal integers (3170724721.py, line 18)

In [15]:
rf_oof["probability"].describe(
    percentiles=[0.10,0.25,0.50,0.75,0.90]
)

count    2520.000000
mean        0.404280
std         0.074426
min         0.227574
10%         0.302056
25%         0.345490
50%         0.409492
75%         0.459733
90%         0.497561
max         0.598720
Name: probability, dtype: float64

In [16]:
bins = [
    -np.inf,
    0.30,
    0.35,
    0.40,
    0.45,
    0.50,
    0.55,
    np.inf
]

labels = [
    "<30%",
    "30-35%",
    "35-40%",
    "40-45%",
    "45-50%",
    "50-55%",
    "55%+"
]

rf_oof["prob_bucket"] = pd.cut(
    rf_oof["probability"],
    bins=bins,
    labels=labels
)

In [17]:
bucket_summary = (
    rf_oof
    .groupby("prob_bucket", observed=True)
    .agg(
        observations=("actual", "count"),
        avg_probability=("probability", "mean"),
        actual_success_rate=("actual", "mean")
    )
)

bucket_summary

,observations,avg_probability,actual_success_rate
prob_bucket,,,
<30%,246,0.275927,0.308943
30-35%,432,0.328973,0.354167
35-40%,480,0.375551,0.397917
40-45%,607,0.425285,0.448105
45-50%,521,0.472167,0.485605
50-55%,186,0.521908,0.494624
55%+,48,0.568867,0.541667


In [18]:
thresholds = [0.40, 0.45, 0.50, 0.55]

threshold_results = []

base_rate = rf_oof["actual"].mean()

for threshold in thresholds:
    selected = rf_oof[
        rf_oof["probability"] >= threshold
    ]

    threshold_results.append({
        "threshold": threshold,
        "observations": len(selected),
        "avg_probability": selected["probability"].mean(),
        "actual_success_rate": selected["actual"].mean(),
        "base_rate": base_rate
    })

threshold_summary = pd.DataFrame(threshold_results)

threshold_summary

,threshold,observations,avg_probability,actual_success_rate,base_rate
0,0.40,1362,0.461474,0.472100,0.421825
1,0.45,755,0.490569,0.491391,0.421825
2,0.50,234,0.531540,0.504274,0.421825
3,0.55,48,0.568867,0.541667,0.421825


In [19]:
threshold_summary["lift_pp"] = (
    threshold_summary["actual_success_rate"]
    - threshold_summary["base_rate"]
) * 100

threshold_summary["coverage"] = (
    threshold_summary["observations"]
    / len(rf_oof)
)

threshold_summary

,threshold,observations,avg_probability,actual_success_rate,base_rate,lift_pp,coverage
0,0.40,1362,0.461474,0.472100,0.421825,5.027446,0.540476
1,0.45,755,0.490569,0.491391,0.421825,6.956533,0.299603
2,0.50,234,0.531540,0.504274,0.421825,8.244811,0.092857
3,0.55,48,0.568867,0.541667,0.421825,11.984127,0.019048


In [20]:
threshold = 0.50

fold_threshold_results = []

for fold in sorted(rf_oof["fold"].unique()):
    fold_data = rf_oof[
        rf_oof["fold"] == fold
    ]

    selected = fold_data[
        fold_data["probability"] >= threshold
    ]

    base_rate = fold_data["actual"].mean()
    success_rate = selected["actual"].mean()

    fold_threshold_results.append({
        "fold": fold,
        "observations": len(selected),
        "base_rate": base_rate,
        "success_rate": success_rate,
        "lift_pp": (success_rate - base_rate) * 100
    })

fold_threshold_summary = pd.DataFrame(
    fold_threshold_results
)

fold_threshold_summary

,fold,observations,base_rate,success_rate,lift_pp
0,1,5,0.428571,0.400000,-2.857143
1,2,28,0.373016,0.607143,23.412698
2,3,39,0.375000,0.461538,8.653846
3,4,63,0.505952,0.634921,12.896825
4,5,99,0.426587,0.414141,-1.244589


In [21]:
fold_threshold_summary["coverage"] = (
    fold_threshold_summary["observations"] / 504
)

fold_threshold_summary

,fold,observations,base_rate,success_rate,lift_pp,coverage
0,1,5,0.428571,0.400000,-2.857143,0.009921
1,2,28,0.373016,0.607143,23.412698,0.055556
2,3,39,0.375000,0.461538,8.653846,0.077381
3,4,63,0.505952,0.634921,12.896825,0.125000
4,5,99,0.426587,0.414141,-1.244589,0.196429


In [22]:
top_pct = 0.10

top10_results = []

for fold in sorted(rf_oof["fold"].unique()):
    fold_data = rf_oof[
        rf_oof["fold"] == fold
    ]

    n_select = int(
        len(fold_data) * top_pct
    )

    selected = fold_data.nlargest(
        n_select,
        "probability"
    )

    base_rate = fold_data["actual"].mean()
    success_rate = selected["actual"].mean()

    top10_results.append({
        "fold": fold,
        "observations": len(selected),
        "base_rate": base_rate,
        "success_rate": success_rate,
        "lift_pp": (
            success_rate - base_rate
        ) * 100
    })

top10_summary = pd.DataFrame(top10_results)

top10_summary

,fold,observations,base_rate,success_rate,lift_pp
0,1,50,0.428571,0.50,7.142857
1,2,50,0.373016,0.46,8.698413
2,3,50,0.375000,0.46,8.500000
3,4,50,0.505952,0.66,15.404762
4,5,50,0.426587,0.48,5.341270


In [23]:
top_percentages = [0.10, 0.20, 0.30]

ranking_results = []

for top_pct in top_percentages:

    for fold in sorted(rf_oof["fold"].unique()):

        fold_data = rf_oof[
            rf_oof["fold"] == fold
        ]

        n_select = int(
            len(fold_data) * top_pct
        )

        selected = fold_data.nlargest(
            n_select,
            "probability"
        )

        base_rate = fold_data["actual"].mean()
        success_rate = selected["actual"].mean()

        ranking_results.append({
            "top_pct": top_pct,
            "fold": fold,
            "observations": len(selected),
            "base_rate": base_rate,
            "success_rate": success_rate,
            "lift_pp": (
                success_rate - base_rate
            ) * 100
        })

ranking_df = pd.DataFrame(ranking_results)

ranking_df

,top_pct,fold,observations,base_rate,success_rate,lift_pp
0,0.1,1,50,0.428571,0.500000,7.142857
1,0.1,2,50,0.373016,0.460000,8.698413
2,0.1,3,50,0.375000,0.460000,8.500000
3,0.1,4,50,0.505952,0.660000,15.404762
4,0.1,5,50,0.426587,0.480000,5.341270
5,0.2,1,100,0.428571,0.520000,9.142857
6,0.2,2,100,0.373016,0.490000,11.698413
7,0.2,3,100,0.375000,0.440000,6.500000
8,0.2,4,100,0.505952,0.590000,8.404762
9,0.2,5,100,0.426587,0.420000,-0.658730


In [24]:
ranking_summary = (
    ranking_df
    .groupby("top_pct")
    .agg(
        mean_success_rate=("success_rate", "mean"),
        mean_lift_pp=("lift_pp", "mean"),
        lift_std=("lift_pp", "std"),
        min_lift_pp=("lift_pp", "min"),
        total_observations=("observations", "sum")
    )
)

ranking_summary

,mean_success_rate,mean_lift_pp,lift_std,min_lift_pp,total_observations
top_pct,,,,,
0.1,0.512000,9.017460,3.814305,5.341270,250
0.2,0.492000,7.017460,4.678272,-0.658730,500
0.3,0.491391,6.956533,2.244601,3.698886,755


In [25]:
ranking_df[
    ranking_df["top_pct"].isin([0.10, 0.30])
][
    [
        "top_pct",
        "fold",
        "observations",
        "base_rate",
        "success_rate",
        "lift_pp"
    ]
]

,top_pct,fold,observations,base_rate,success_rate,lift_pp
0,0.1,1,50,0.428571,0.500000,7.142857
1,0.1,2,50,0.373016,0.460000,8.698413
2,0.1,3,50,0.375000,0.460000,8.500000
3,0.1,4,50,0.505952,0.660000,15.404762
4,0.1,5,50,0.426587,0.480000,5.341270
10,0.3,1,151,0.428571,0.496689,6.811731
11,0.3,2,151,0.373016,0.470199,9.718280
12,0.3,3,151,0.375000,0.456954,8.195364
13,0.3,4,151,0.505952,0.569536,6.358404
14,0.3,5,151,0.426587,0.463576,3.698886


In [26]:
LOOKBACK = 252

rf_oof["rolling_p90"] = (
    rf_oof["probability"]
    .shift(1)
    .rolling(LOOKBACK)
    .quantile(0.90)
)

In [27]:
rf_oof[
    ["probability", "rolling_p90", "actual"]
].iloc[245:265]

,probability,rolling_p90,actual
Date,,,
2013-12-13,0.455726,NaN,1
2013-12-16,0.488478,NaN,1
2013-12-17,0.484057,NaN,1
2013-12-18,0.419715,NaN,1
2013-12-19,0.375904,NaN,1
2013-12-20,0.342363,NaN,1
2013-12-23,0.334224,NaN,1
2013-12-24,0.267677,0.453179,0
2013-12-26,0.299633,0.453179,0


In [29]:
adaptive_eval = rf_oof.dropna(
    subset=["rolling_p90"]
).copy()

adaptive_eval["signal"] = (
    adaptive_eval["probability"]
    >= adaptive_eval["rolling_p90"]
).astype(int)

adaptive_eval[
    ["probability", "rolling_p90", "signal", "actual"]
].head(10)

,probability,rolling_p90,signal,actual
Date,,,,
2013-12-24,0.267677,0.453179,0,0
2013-12-26,0.299633,0.453179,0,0
2013-12-27,0.308535,0.451725,0,0
2013-12-30,0.332073,0.451110,0,0
2013-12-31,0.315538,0.451110,0,0
2014-01-02,0.386389,0.451110,0,0
2014-01-03,0.413654,0.451110,0,0
2014-01-06,0.458345,0.451110,1,1
2014-01-07,0.393664,0.451725,0,0


In [30]:
selected = adaptive_eval[
    adaptive_eval["signal"] == 1
]

base_rate = adaptive_eval["actual"].mean()
signal_success_rate = selected["actual"].mean()
coverage = len(selected) / len(adaptive_eval)

print("Eligible days:", len(adaptive_eval))
print("Signals:", len(selected))
print("Coverage:", coverage)
print("Base rate:", base_rate)
print("Signal success rate:", signal_success_rate)
print(
    "Lift (pp):",
    (signal_success_rate - base_rate) * 100
)

Eligible days: 2268
Signals: 286
Coverage: 0.12610229276895943
Base rate: 0.41798941798941797
Signal success rate: 0.486013986013986
Lift (pp): 6.802456802456803


In [31]:
adaptive_fold_results = []

for fold in sorted(adaptive_eval["fold"].unique()):

    fold_data = adaptive_eval[
        adaptive_eval["fold"] == fold
    ]

    selected = fold_data[
        fold_data["signal"] == 1
    ]

    base_rate = fold_data["actual"].mean()
    success_rate = selected["actual"].mean()

    adaptive_fold_results.append({
        "fold": fold,
        "eligible_days": len(fold_data),
        "signals": len(selected),
        "coverage": len(selected) / len(fold_data),
        "base_rate": base_rate,
        "success_rate": success_rate,
        "lift_pp": (
            success_rate - base_rate
        ) * 100
    })

adaptive_fold_summary = pd.DataFrame(
    adaptive_fold_results
)

adaptive_fold_summary

,fold,eligible_days,signals,coverage,base_rate,success_rate,lift_pp
0,1,252,28,0.111111,0.400794,0.500000,9.920635
1,2,504,56,0.111111,0.373016,0.500000,12.698413
2,3,504,76,0.150794,0.375000,0.421053,4.605263
3,4,504,47,0.093254,0.505952,0.595745,8.979230
4,5,504,79,0.156746,0.426587,0.468354,4.176713


In [32]:
quantiles = [0.70, 0.80, 0.90]

adaptive_results = []

for q in quantiles:

    threshold_col = f"rolling_p{int(q * 100)}"

    rf_oof[threshold_col] = (
        rf_oof["probability"]
        .shift(1)
        .rolling(252)
        .quantile(q)
    )

    temp = rf_oof.dropna(
        subset=[threshold_col]
    ).copy()

    temp["signal"] = (
        temp["probability"] >= temp[threshold_col]
    ).astype(int)

    selected = temp[temp["signal"] == 1]

    base_rate = temp["actual"].mean()
    success_rate = selected["actual"].mean()

    adaptive_results.append({
        "quantile": q,
        "signals": len(selected),
        "coverage": len(selected) / len(temp),
        "success_rate": success_rate,
        "lift_pp": (
            success_rate - base_rate
        ) * 100
    })

adaptive_comparison = pd.DataFrame(
    adaptive_results
)

adaptive_comparison

,quantile,signals,coverage,success_rate,lift_pp
0,0.7,750,0.330688,0.480000,6.201058
1,0.8,534,0.235450,0.473783,5.579335
2,0.9,286,0.126102,0.486014,6.802457


In [33]:
quantiles_to_check = [0.70, 0.90]

adaptive_fold_compare = []

for q in quantiles_to_check:

    threshold_col = f"rolling_p{int(q * 100)}"

    temp = rf_oof.dropna(
        subset=[threshold_col]
    ).copy()

    temp["signal"] = (
        temp["probability"] >= temp[threshold_col]
    ).astype(int)

    for fold in sorted(temp["fold"].unique()):

        fold_data = temp[
            temp["fold"] == fold
        ]

        selected = fold_data[
            fold_data["signal"] == 1
        ]

        base_rate = fold_data["actual"].mean()
        success_rate = selected["actual"].mean()

        adaptive_fold_compare.append({
            "quantile": q,
            "fold": fold,
            "signals": len(selected),
            "coverage": len(selected) / len(fold_data),
            "base_rate": base_rate,
            "success_rate": success_rate,
            "lift_pp": (
                success_rate - base_rate
            ) * 100
        })

adaptive_fold_compare = pd.DataFrame(
    adaptive_fold_compare
)

adaptive_fold_compare

,quantile,fold,signals,coverage,base_rate,success_rate,lift_pp
0,0.7,1,81,0.321429,0.400794,0.506173,10.537919
1,0.7,2,159,0.315476,0.373016,0.465409,9.239293
2,0.7,3,174,0.345238,0.375000,0.465517,9.051724
3,0.7,4,142,0.281746,0.505952,0.549296,4.334339
4,0.7,5,194,0.384921,0.426587,0.443299,1.671167
5,0.9,1,28,0.111111,0.400794,0.500000,9.920635
6,0.9,2,56,0.111111,0.373016,0.500000,12.698413
7,0.9,3,76,0.150794,0.375000,0.421053,4.605263
8,0.9,4,47,0.093254,0.505952,0.595745,8.979230
9,0.9,5,79,0.156746,0.426587,0.468354,4.176713


In [34]:
print("Signal candidates after walk-forward research:")
print("P70: adaptive 70th percentile threshold")
print("P90: adaptive 90th percentile threshold")

print("\nFinal test remains untouched:")
print(
    final_test_df.index.min(),
    "→",
    final_test_df.index.max()
)

Signal candidates after walk-forward research:
P70: adaptive 70th percentile threshold
P90: adaptive 90th percentile threshold

Final test remains untouched:
2023-01-03 00:00:00 → 2026-08-21 00:00:00
